In [ ]:
import os
import sys
# 假设 dataset0 文件夹在当前工作目录下
module_path = os.path.abspath(os.path.join('./Agrobot Dataset/dataset0'))
if module_path not in sys.path:
    sys.path.append(module_path)
print(os.getcwd()) 
module_path = os.path.abspath(os.path.join('./Neural Kalman IMU GNSS Fusion'))
if module_path not in sys.path:
    sys.path.append(module_path)
print(os.getcwd()) 

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("使用的设备：", device)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="0"
import pandas as pd
import numpy as np
from gtda.time_series import SlidingWindow
from math import atan2, pi, sqrt, cos, sin, floor
import tensorflow as tf
from tensorflow.python.keras.backend import set_session
config = tf.compat.v1.ConfigProto() 
config.gpu_options.allow_growth = True  
config.log_device_placement = True  
sess2 = tf.compat.v1.Session(config=config)
set_session(sess2)  
from tensorflow.keras.layers import Dense, MaxPooling1D, Flatten, Reshape
from tensorflow.keras import Input, Model
from tensorflow.keras.callbacks import ModelCheckpoint
import tensorflow.keras.backend as K
from tensorflow.keras.models import load_model
from sklearn.metrics import mean_squared_error
from scipy.stats import uniform
import pickle
import csv
import random
import itertools
import matplotlib.pyplot as plt
import math
import time
from data_utils_0 import *
from traj_utils import *
from torchviz import make_dot
import tensorboard
from tqdm import tqdm
import torch.nn.functional as F
from traj_utils import *
from neural_ekf import *

# 定义脉冲神经元

In [ ]:
# 把每个输出值转为0~1的值
class IFNeuron(nn.Module):
    def __init__(self, threshold=1.0, decay=0.9):
        super(IFNeuron, self).__init__()
        self.threshold = threshold  # 膜电位阈值
        self.decay = decay  # 膜电位衰减系数
        self.mem = None  # 初始化膜电位(memory)为None

    def forward(self, x):
        if self.mem is None:
            self.mem = torch.zeros_like(x)  # 初始化膜电位为与输入张量x相同形状的零张量
        device = x.device
        self.mem.to(device) # 将膜电位移动到与输入张量x相同的设备上
        self.mem = self.mem * self.decay + x  # 更新膜电位,加入衰减
        spikes = (self.mem >= self.threshold).float()  # 判断是否超过阈值，生成脉冲
        self.mem[self.mem >= self.threshold] -= self.threshold  # 对超过阈值的膜电位进行重置
        return spikes

    def reset(self):
        self.mem = None  # 重置膜电位


# 定义泊松编码器

In [ ]:
class PoissonEncoder(nn.Module): #自定义泊松编码器
    def __init__(self):
        super(PoissonEncoder, self).__init__()

    def forward(self, x, time_steps):
        # x的形状为 (batch_size, in_channels, timesteps)
        shape = x.shape + (time_steps,)
        return (torch.rand(shape) < x.unsqueeze(-1))


# 定义TCN网络层

In [ ]:
# 定义TCN网络层
class TCNLayer(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilations, dropout_rate, skip_connect, T):
        super(TCNLayer, self).__init__()
        self.out_channels = out_channels
        self.skip_connect = skip_connect
        self.T = T #循环周期 
        self.layers = nn.ModuleList()
        
        # 叠加多个TCN层
        for dilation in dilations:
            self.layers.append(nn.Sequential(
                nn.Conv1d(in_channels, self.out_channels, kernel_size, padding=(kernel_size-1)//2 * dilation, dilation=dilation),
                nn.BatchNorm1d(self.out_channels),
                nn.Dropout(dropout_rate),
                IFNeuron() #脉冲神经元的数量为out_channels
            ))
            in_channels = out_channels  # Update in_channels for the next layer


    def forward(self, x):
        batch_size = x.size(0)
        time_steps = x.size(2)
        device = x.device
        out_fr = torch.zeros(batch_size, self.out_channels, time_steps, device=device) #初始化零向量
        self.residual = torch.zeros(batch_size, self.out_channels, time_steps, device=device) #初始化残差向量
          # Compute the residual connection
        for t in range(self.T):
            o = x
            i = 0 
            for layer in self.layers: #在TCN网络层中正向传播
                o = layer(o)
                if (self.skip_connect is True) and (i%2 == 0):
                    self.residual += o #得到奇数层的残差
                i += 1
            o += self.residual #将TCN网络的输出与残差相加
            out_fr += o #累加TCN网络的输出，得到累加值out_fr
        out_fr = out_fr/(4*self.T) #计算T内的发射频率
        return out_fr #输出格式为（batch_size, out_channels)
    
    def reset(self):
        for layer in self.layers:
            layer[3].reset()


# 定义完整的TCN模型

In [ ]:
class TCNModel(nn.Module):  
    def __init__(self, in_channels, out_channels, kernel_size, dilations, dropout_rate, skip_connect, T):
        super(TCNModel, self).__init__()  # 调用父类 nn.Module 的构造函数
        self.T = T
        self.tcn = TCNLayer(in_channels, out_channels, kernel_size, dilations, dropout_rate, skip_connect, self.T)
        self.out_channels = out_channels
        # 1d平均池化层
        self.meanpool = nn.AvgPool1d(4, 4)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(100*self.out_channels//4, 32)
        self.opx = nn.Linear(32, 1)
        self.opy = nn.Linear(32, 1)

        
    def forward(self, x):
        x = self.tcn(x) #输出为(batch_size, 32, 100),得到的是tcn输出的out_fr(tensor)
        x = self.meanpool(x)
        x = torch.flatten(x, start_dim=1) #将x展平为一维
        x = self.fc(x) #通过一层全连接层
        #前向传播计算x值
        output_x = self.opx(x)
        #前向传播计算y值
        output_y = self.opy(x)
    
        return [output_x, output_y]
    
    def reset(self):
        self.tcn.reset()


# 读取数据

In [ ]:
window_size = 100
stride = 20

f = './Agrobot Dataset/dataset0/'
X_train,Y_Pos_train, Physics_Vec_train, x_vel_train, y_vel_train, x0_list_train, y0_list_train, size_of_each_train = import_agrobot_dataset_p1(dataset_folder=f, type_flag=1, window_size=window_size, stride=stride)
P = np.repeat(Physics_Vec_train,window_size).reshape((Physics_Vec_train.shape[0],window_size,1)) #得到训练数据集
X_train = np.concatenate((X_train,P),axis=2) #按列拼接得到第十个频域特征



X_test,Y_Pos_test, Physics_Vec_test, x_vel_test, y_vel_test, x0_list_test, y0_list_test, size_of_each_test= import_agrobot_dataset_p1(type_flag = 2, dataset_folder=f,window_size=window_size, stride=stride)
P_test = np.repeat(Physics_Vec_test,window_size).reshape((Physics_Vec_test.shape[0],window_size,1)) #得到测试数据集
X_test = np.concatenate((X_test,P_test),axis=2)


X_test = torch.tensor(X_test)
X_train = torch.tensor(X_train)
x_vel_test = torch.tensor(x_vel_test)
x_vel_train = torch.tensor(x_vel_train)
y_vel_test = torch.tensor(y_vel_test)
y_vel_train = torch.tensor(y_vel_train)
print(X_train.dtype)
X_train = torch.transpose(X_train, 1, 2)
X_test = torch.transpose(X_test, 1, 2)
print(X_test.shape)

In [ ]:
kernel_size = 5
dilations = [1,2,4,8,16,32,64]
dropout_rate = 0.0
skip_connect = True
in_channels = 10
out_channels =32
T = 3


batch_size, timesteps, input_dim = 256, window_size, X_train.shape[2]


model = TCNModel(in_channels, out_channels, kernel_size, dilations, dropout_rate, skip_connect, T).to(device)
model.float()
best_val_loss = float('inf') 
checkpoint = './extensions/PreTrained_Models/Agrobot_First_spike.pth'

# 创建 TensorDataset 和 DataLoader
dataset = TensorDataset(X_train, x_vel_train, y_vel_train) #将训练数据转为数据集
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True) #规定取样方式

# 初始化损失函数和优化器
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# 训练循环
num_epochs = 300
start_time = time.time()  # 记录训练开始时间
for epoch in tqdm(range(num_epochs)):
    model.train()
    loss_epoch = 0
    epoch_start_time = time.time()
    for i,(features, label_x, label_y) in enumerate(train_loader):
        features, label_x, label_y = features.to(device).float(), label_x.to(device).float(), label_y.to(device).float()
        output_x, output_y = model(features)

        optimizer.zero_grad()
        loss_x = criterion(output_x, label_x)
        loss_x.backward(retain_graph=True)  # 反向传播
        optimizer.step() # 更新参数

        optimizer.zero_grad()
        loss_y = criterion(output_y, label_y)
        loss_y.backward()  # 反向传播
        optimizer.step() # 更新参数

        loss_epoch = loss_epoch + loss_x + loss_y

        model.reset()  # 在每个批次之后重置脉冲神经元的状态


        epoch_end_time = time.time()  # 记录每个 epoch 的结束时间
        epoch_duration = epoch_end_time - epoch_start_time  # 计算每个 epoch 所花费的时间
        
        # 计算剩余时间
        elapsed_time = epoch_end_time - start_time  # 已经过的总时间
        avg_epoch_time = elapsed_time / (epoch + 1)  # 平均每个 epoch 的时间
        remaining_epochs = num_epochs - (epoch + 1)  # 剩余的 epoch 数
        estimated_remaining_time = remaining_epochs * avg_epoch_time  # 估计的剩余时间
        
    print(f'Elapsed Time: {elapsed_time:.2f} seconds, Estimated Remaining Time: {estimated_remaining_time:.2f} seconds')

    # 保存验证损失最低的模型
    if loss_epoch < best_val_loss:
        
        best_val_loss = loss_epoch
        # 保存多个对象
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'epoch': epoch,
            'loss': best_val_loss
        }, checkpoint)
        print(f'Best model saved at epoch {epoch+1}')
        
end_time = time.time()  # 记录训练结束时间
total_duration = end_time - start_time  # 计算总训练时间
print(f'Total Training Time: {total_duration:.2f} seconds')


In [ ]:
# 保存模型
torch.save(model, './extensions/PreTrained_Models\Agrobot_First_spike.pkl')  

# 定义模型评估函数

In [ ]:
#模型评估函数
def evaluate_model(model, test_loader):
    model.eval()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    
    total_mse = 0.0
    total_mae = 0.0
    total_samples = 0
    
    with torch.no_grad():
        for X_test_batch, x_vel_test_batch, y_vel_test_batch in test_loader:
            X_test_batch, x_vel_test_batch, y_vel_test_batch = X_test_batch.to(device).float(), x_vel_test_batch.to(device).float(), y_vel_test_batch.to(device).float()
            predict_x, predict_y = model(X_test_batch)
            
            mse_x = F.mse_loss(predict_x, x_vel_test_batch, reduction='sum').item()
            mse_y = F.mse_loss(predict_y, y_vel_test_batch, reduction='sum').item()
            mse = mse_x + mse_y
            mae_x = F.l1_loss(predict_x, x_vel_test_batch, reduction='sum').item()
            mae_y = F.l1_loss(predict_y, y_vel_test_batch, reduction='sum').item()
            mae = mae_x + mae_y
            
            total_mse += mse
            total_mae += mae
            total_samples += len(X_test_batch)
    
    mse = total_mse / total_samples
    rmse = torch.sqrt(torch.tensor(mse)).item()
    mae = total_mae / total_samples
    
    return mse, rmse, mae

In [ ]:
#对模型进行评估

# 创建测试数据集和数据加载器
test_dataset = TensorDataset(X_test, x_vel_test, y_vel_test)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, drop_last=True)

# 将模型设置为评估模式
model.eval()

mse, rmse, mae = evaluate_model(model, test_loader)
print(f"Test Set MSE: {mse:.4f}")
print(f"Test Set RMSE: {rmse:.4f}")
print(f"Test Set MAE: {mae:.4f}")

In [ ]:
torch.save(model, './extensions/PreTrained_Models/Agrobot_First_model.pth')

# 计算模型的ATE,RTE

## 重新定义GT_pos_generator函数

In [ ]:
def GT_pos_generator(GT_vel_x, GT_vel_y, size_of_each, 
                   x0_list, y0_list, window_size, stride,file_idx):  
    if (file_idx == 0):
        x_vel_test_sel = GT_vel_x[0:size_of_each[0]] #将真实坐标值作为测试数据集
        y_vel_test_sel = GT_vel_y[0:size_of_each[0]]
    elif (file_idx == 1):
        x_vel_test_sel = GT_vel_x[np.sum(size_of_each[0]):np.sum(size_of_each[0:file_idx+1])]
        y_vel_test_sel = GT_vel_y[np.sum(size_of_each[0]):np.sum(size_of_each[0:file_idx+1])]

    else:
        x_vel_test_sel = GT_vel_x[np.sum(size_of_each[0:file_idx]):np.sum(size_of_each[0:file_idx+1])] #根据文件序号区分对应的速度真值数据
        y_vel_test_sel = GT_vel_y[np.sum(size_of_each[0:file_idx]):np.sum(size_of_each[0:file_idx+1])]
    
    pointx = []
    pointy = []
    Lx =  x0_list[file_idx]
    Ly = y0_list[file_idx]
    for j in range(len(x_vel_test_sel)):
        Lx = Lx + (x_vel_test_sel[j]/(((window_size-stride)/stride)))
        Ly = Ly + (y_vel_test_sel[j]/(((window_size-stride)/stride)))    
        pointx.append(Lx)
        pointy.append(Ly)   
    Gvx = pointx
    Gvy = pointy
    
    return Gvx, Gvy #返回真值数据点

## 重新定义model_pos_generator函数

In [ ]:

def model_pos_generator(net_inp_mat, size_of_each, 
                   x0_list, y0_list, window_size, stride,file_idx,my_model):
    
    if (file_idx == 0):
        cur_inp = net_inp_mat[0:size_of_each[0],:,:]
    elif (file_idx == 1):
        cur_inp = net_inp_mat[np.sum(size_of_each[0]):np.sum(size_of_each[0:file_idx+1]),:,:]
        
    else:
        cur_inp = net_inp_mat[np.sum(size_of_each[0:file_idx]):np.sum(size_of_each[0:file_idx+1]),:,:]    
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    cur_inp = cur_inp.to(device).float() 
    
    pos_dataset = TensorDataset(cur_inp)
    pos_loader = DataLoader(pos_dataset, batch_size=256, shuffle=False, drop_last=True)

    pointx = []
    pointy = []
    Lx =  x0_list[file_idx]
    Ly = y0_list[file_idx]
    for cur_batch in pos_loader:
        cur_batch = cur_batch[0].to(device).float()
        y_pred = my_model(cur_batch)
        for j in range(len(cur_batch)):
            Lx = Lx + (y_pred[0][j][0]/(((window_size-stride)/stride)))
            Ly = Ly + (y_pred[1][j][0]/(((window_size-stride)/stride)))
            pointx.append(Lx)
            pointy.append(Ly)
    Pvx = pointx
    Pvy = pointy   
    
    return Pvx, Pvy

## 重新定义Cal_TE函数

In [ ]:

def Cal_TE(Gvx, Gvy, Pvx, Pvy, sampling_rate=100,window_size=200,stride=10,length=None):
    
    if length==None:
        length = len(Pvx)
        
    distance = []
    
    for i in range(length):
        d = ((Gvx[i]-Pvx[i])*(Gvx[i]-Pvx[i])) + ((Gvy[i]-Pvy[i])*(Gvy[i]-Pvy[i])) #计算轨迹偏差的平方
        d = math.sqrt(d)
        distance.append(d) #将轨迹的偏差集合到一个列表中
    
    mean_distance = sum(distance)/len(distance) #求总体轨迹平均偏差值即ATE
    ate = mean_distance
    at_all = distance #各点偏差的列表
    
    n_windows_one_min= int(((sampling_rate*60)-window_size)/stride)
    distance = []
    if(n_windows_one_min < length):
        for i in range(n_windows_one_min):
            d = ((Gvx[i]-Pvx[i])*(Gvx[i]-Pvx[i])) + ((Gvy[i]-Pvy[i])*(Gvy[i]-Pvy[i]))
            d = math.sqrt(d)
            distance.append(d) #distance容纳一分钟内所有的相对偏差
        rte = sum(distance)/len(distance) #求平均值得到ATE
    else:
        rte=ate*(n_windows_one_min/length) #根据比率求得RTE
    
    rt_all = distance
    return ate, rte, at_all, rt_all #返回平均的ATE、RTE，轨迹总的ATE、RTE

In [ ]:
#对新训练的模型测试部分提取轨迹与计算ATE, RTE

ATE = []
RTE = []
ATE_dist = []
RTE_dist = []
for i in range(len(size_of_each_test)):
    Pvx, Pvy = model_pos_generator(X_test, size_of_each_test, 
                   x0_list_test, y0_list_test, window_size, stride,i,model)   #i指文件序号 返回轨迹数据
    Gvx, Gvy = GT_pos_generator(x_vel_test,y_vel_test,size_of_each_test, 
                                x0_list_test, y0_list_test, window_size, stride,i)  #返回轨迹真值数据
    
    at, rt, at_all, rt_all = Cal_TE(Gvx, Gvy, Pvx, Pvy,
                                    sampling_rate=100,window_size=window_size,stride=stride) #返回ATE,RTE，轨迹中所有点的偏差，一分钟内轨迹的点的偏差
    ATE.append(at)
    RTE.append(rt)
    ATE_dist.append(Cal_len_meters(Gvx, Gvy)) #计算轨迹的总距离
    RTE_dist.append(Cal_len_meters(Gvx, Gvy, 600)) #计算一分钟内轨迹的总距离
    print('ATE, RTE, Trajectory Length, Trajectory Length (60 seconds)',ATE[i],RTE[i],ATE_dist[i],RTE_dist[i])
    
print('Median ATE and RTE', np.median(ATE),np.median(RTE))
print('Mean ATE and RTE', np.mean(ATE),np.mean(RTE))
print('STD ATE and RTE', np.std(ATE),np.std(RTE))

In [ ]:
#画真实轨迹图与预测轨迹图
fused_pos_x, fused_pos_y, GPS_x, GPS_y =  neural_ekf_gnss_imu(X_train, x_vel_train,y_vel_train, 
            size_of_each_train,
            x0_list_train, y0_list_train,0,window_size,stride,5,
             model) #获取指定数据集的N_EKF的预测结果
act_x,act_y =  GT_pos_generator(x_vel_train, y_vel_train, size_of_each_train, 
               x0_list_train, y0_list_train, window_size, stride,0) #获取指定数据集的真实位置数据

In [ ]:
a = 0
b = 3000
plt.plot(act_x[a:b],act_y[a:b],label='Ground Truth',linestyle='-') #画真实位置轨迹
plt.plot(fused_pos_x[a:b],fused_pos_y[a:b],label='Neurl-KF',linestyle='-') #画预测位置轨迹
plt.scatter(GPS_x[math.ceil(a/5):math.ceil(b/5)],GPS_y[math.ceil(a/5):math.ceil(b/5)],
            marker='.',label='GPS only') #画出GPS测试轨迹以作对比
plt.xlim([-2,1.5])
plt.ylim([-2,2])
plt.grid('minor')
plt.xlabel('East (m)')
plt.ylabel('North (m)')
plt.title('Phase 1, with GPS')
plt.legend(loc='best')

# 模型可视化

In [ ]:
kernel_size = 5
dilations = [1,2,4,8,16,32,64]
dropout_rate = 0.0
skip_connect = True
in_channels = 10
out_channels =32
T = 5

batch_size, timesteps, input_dim = 256, window_size, X_train.shape[2]

test_dataset = TensorDataset(X_test, x_vel_test, y_vel_test)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

model = TCNModel(in_channels, out_channels, kernel_size, dilations, dropout_rate, skip_connect, T).to(device)
# 初始化损失函数和优化器
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 加载保存的检查点
checkpoint = torch.load('./extensions/PreTrained_Models/Agrobot_First_spike.pth')

# 加载模型和优化器的状态字典
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

torch.save(model, './extensions/PreTrained_Models/Agrobot_First_spike_1.pth')
print(model)

model.eval()
# 输出训练的特征维度与标签维度
for features, label_x, label_y in test_loader:
    
    features = features.float().to(device)
        
    y = model(features)
    dot = make_dot(y, params=dict(model.named_parameters()))

    break

# 数据观察

In [ ]:
X_train = torch.tensor(X_train)
x_vel_train = torch.tensor(x_vel_train)
y_vel_train = torch.tensor(y_vel_train)
dataset = TensorDataset(X_train, x_vel_train, y_vel_train)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# 输出训练的特征维度与标签维度
for features, label_x, label_y in train_loader:
    print('feature shape:', features.shape)
    print('label_x shape:', label_x.shape)
    print('label_y shape:', label_y.shape)
    break

In [ ]:
print(Y_Pos_train.shape)
print(x_vel_train.shape)
print(y_vel_train.shape)
# 输出训练数据的列数
print(X_train.size(2))

In [ ]:
# 输出示例样本
import torch

# 假设 batch_size 为 4，时间步数为 50，神经元数量为 10
batch_size = 4
time_steps = 50
num_neurons = 10

# 随机生成脉冲神经网络的输出 (batch_size, time_steps, num_neurons)
spike_output = torch.randint(0, 2, (batch_size, time_steps, num_neurons))

print("Spike Output Shape:", spike_output.shape)
print("Spike Output:\n", spike_output)

In [ ]:
print('x_max:', torch.mean(torch.abs(torch.tensor(x_vel_test))))
print('y_max:', torch.mean(torch.abs(torch.tensor(y_vel_test))))